# Stage 1 — Topic → Theme (temporal) with **BERTrend**

Roadmap goal: turn per-window BERTopic clusters into **themes with a stable ID over time** (continuation / merge / split / birth), build a **theme-intensity time series**, and label lifecycle stage.

[BERTrend](https://github.com/rte-france/BERTrend) trains a BERTopic model **per time slice**, merges topics across slices with a cosine threshold to form cumulative themes, then computes a **popularity (intensity)** signal per theme with exponential decay — classifying each as `noise` / `weak signal` / `strong signal`.

**This run (v2)** incorporates the review findings from the H1 prototype:

1. **Wider window** — full **2023 + 2024** (~52 bi-weekly slices) instead of one half-year.
2. **Cleaner inputs** — strip wire-source prefixes, drop earnings/announcement boilerplate, and extend the stopword list, so junk clusters (`inc, plc, results, ceo, …`) stop dominating.
3. **Intensity *rate*** — add a first-difference / per-slice doc-count signal so themes show birth→peak→decay (needed for the lead–lag test), not just cumulative attention.
4. **Honest sector map** — calibrate the cosine threshold empirically *and* add a complementary mapping via the `DerivedTickersId` already in the news.
5. **Native BERTrend dashboards** — three HTML artifacts mirroring the official demos:
   - [`topic_analysis`](https://github.com/rte-france/BERTrend/tree/main/bertrend/demos/topic_analysis) — per-slice topic exploration
   - [`weak_signals`](https://github.com/rte-france/BERTrend/tree/main/bertrend/demos/weak_signals) — signal evolution + merge Sankey + emergence
   - [`prospective_demo`](https://github.com/rte-france/BERTrend/tree/main/bertrend/bertrend_apps/prospective_demo) — point-in-time signal dashboard

**Stage-1 gate (later):** does our theme-intensity series *lead* the ETF cohort's price peak (which leads flows)? See `docs/roadmap.md`.

## 0. Environment

BERTrend is part of the main project env (added to `pyproject.toml`; `pandas` pinned `>=2.2,<3.0.0`). To (re)install: `uv sync` from `code/`. Use the project `.venv` kernel.

> **Note:** BERTrend creates a base directory for its models/cache (defaults to `~/.bertrend`). The first code cell sets `BERTREND_BASE_DIR` to a folder inside `notebooks/output/` **before** importing BERTrend, so nothing is written to your home dir.

In [1]:
# Keep BERTrend's base dir inside the project (must be set BEFORE importing bertrend).
import os
from pathlib import Path

_PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ["BERTREND_BASE_DIR"] = str(_PROJECT_ROOT / "notebooks" / "output" / "bertrend_base")

import bertrend  # noqa: E402
import pandas as pd  # noqa: E402

print("bertrend base dir:", bertrend.BASE_PATH)
print("pandas:           ", pd.__version__, "(BERTrend requires <3.0.0)")

2026-06-08 16:12:14.023 | WARNING  | bertrend:<module>:22 - Failed to load .env file


bertrend base dir: /Users/federicocinus/Progetti - Local/ThematicTrading/code/notebooks/output/bertrend_base
pandas:            2.3.3 (BERTrend requires <3.0.0)


## 1. Config & imports

In [2]:
from __future__ import annotations

import lzma
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import torch
from sentence_transformers import SentenceTransformer

from bertrend.BERTrend import BERTrend
from bertrend.BERTopicModel import BERTopicModel
from bertrend.utils.data_loading import (
    group_by_days,
    TEXT_COLUMN,
    TIMESTAMP_COLUMN,
    DOCUMENT_ID_COLUMN,
    SOURCE_COLUMN,
    URL_COLUMN,
)

# --- paths ---
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
ICB_PATH = PROJECT_ROOT / "notebooks" / "input" / "icb-structure-and-definitions.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "output"
MODELS_DIR = PROJECT_ROOT / "notebooks" / "output" / "bertrend_models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL = "FinLang/finance-embeddings-investopedia"
RANDOM_SEED = 42

# --- temporal window: full 2023 + 2024 ---
YEARS = [2023, 2024]
DATE_START = f"{YEARS[0]}-01-01T00:00:00+00:00"
DATE_END = f"{YEARS[-1] + 1}-01-01T00:00:00+00:00"
TAG = f"{YEARS[0]}_{YEARS[-1]}"               # output-file suffix
GRANULARITY_DAYS = 14                          # one BERTopic model every 2 weeks
WINDOW_SIZE = 28                               # rolling window (days) for signal classification
SAMPLE_N = 60_000                              # cap total headlines for tractable embedding

# --- per-slice BERTopic params ---
MIN_TOPIC_SIZE = 25                            # larger clusters -> fewer junk micro-topics
MIN_SAMPLES = 5
MIN_SIMILARITY = 0.7                           # cosine threshold for merging themes across slices

DEVICE = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Device:      {DEVICE}")
print(f"Window:      {DATE_START[:10]} → {DATE_END[:10]}  ({GRANULARITY_DAYS}d slices)")
print(f"Year files:  {[str(RAW_DIR / f'raw_news_{y}.csv.xz') for y in YEARS]}")

Device:      mps
Window:      2023-01-01 → 2025-01-01  (14d slices)
Year files:  ['/Users/federicocinus/Progetti - Local/ThematicTrading/code/data/raw/raw_news_2023.csv.xz', '/Users/federicocinus/Progetti - Local/ThematicTrading/code/data/raw/raw_news_2024.csv.xz']


## 2. Load & clean headlines → BERTrend schema

Polars does not auto-decompress `.xz`, so we open each year file with `lzma` and concatenate. Then we **clean aggressively** (the H1 run was dominated by wire boilerplate):

- `strip_colon_prefix` — drop short wire-source prefixes (`BSECorpAnn:`, `Reuters:`) but keep real content.
- `strip_dates` — remove dates / bare numbers.
- **Boilerplate blocklist** — drop earnings / announcement / governance wire copy (`half-year-report`, `dividend`, `appoints`, `total voting rights`, …) and headlines with < 4 words.

BERTrend expects a pandas frame with `timestamp` + `text`; we also keep `DerivedTickersId` (for the sector/ticker map), `source`, `url`, `document_id`. We **sort by time and reset the index** so embedding row *i* ↔ `document_id == i` (BERTrend slices via `embeddings[group.index]`).

In [3]:
DATE_PATTERNS = (
    r"\b\d{4}[/\-]\d{1,2}[/\-]\d{1,2}\b",
    r"\b\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}\b",
    r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\.?\s+\d{1,2},?\s+\d{4}\b",
    r"\bq[1-4]\s+\d{4}\b",
)

# Earnings / announcement / governance wire boilerplate — the noise clusters in the H1 run.
BOILERPLATE_RE = re.compile(
    r"\b(half[- ]year|annual report|interim (results|report)|quarterly report|"
    r"results? (announcement|publication)|dividend|interim dividend|agm|egm|"
    r"notice of|appoint(s|ed|ment)?|resignation|board change|"
    r"transaction in own shares|total voting rights|director/pdmr|pdmr|"
    r"holding\(s\) in company|net asset value|prospectus|earnings call|"
    r"conference call|webcast|trading update|block listing)\b",
    re.IGNORECASE,
)


def strip_colon_prefix(text: str) -> str:
    """Drop short wire-source prefixes like 'BSECorpAnn:'. Applied twice for stacked prefixes."""
    for _ in range(2):
        if not text or ":" not in text:
            return text
        prefix, _, rest = text.partition(":")
        prefix, rest = prefix.strip(), rest.strip()
        if not prefix or not prefix[0].isalpha() or not rest:
            return text
        if len(prefix) > 30 or len(prefix.split()) > 4:
            return text
        text = rest
    return text


def strip_dates(text: str) -> str:
    for pat in DATE_PATTERNS:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)
    text = re.sub(r":\s*\d+\b", ":", text)
    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r":\s*$", "", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_headline(text: str) -> str:
    return strip_dates(strip_colon_prefix(text))


# --- load every year file, filter to the window, concat, dedupe ---
WANT_COLS = ["Headline", "CaptureTime", "WireName", "DerivedTickersId"]
frames = []
for yr in YEARS:
    p = RAW_DIR / f"raw_news_{yr}.csv.xz"
    with lzma.open(p, "rb") as f:
        frames.append(
            pl.scan_csv(f, infer_schema_length=10_000)
            .select(WANT_COLS)
            .with_columns(pl.col("CaptureTime").str.to_datetime(time_zone="UTC", strict=False))
            .filter(
                (pl.col("CaptureTime") >= pl.lit(DATE_START).str.to_datetime(time_zone="UTC"))
                & (pl.col("CaptureTime") < pl.lit(DATE_END).str.to_datetime(time_zone="UTC"))
                & pl.col("Headline").is_not_null()
                & (pl.col("Headline").str.len_chars() > 0)
            )
            .collect()
        )
corpus = pl.concat(frames).unique(subset=["Headline"])
print(f"Unique headlines in window: {corpus.height:,}")

if corpus.height > SAMPLE_N:
    corpus = corpus.sample(n=SAMPLE_N, seed=RANDOM_SEED, shuffle=True)

# --- to pandas + clean ---
df = corpus.to_pandas()
df = df.rename(columns={"Headline": TEXT_COLUMN, "CaptureTime": TIMESTAMP_COLUMN, "WireName": SOURCE_COLUMN})
n0 = len(df)

# drop boilerplate on the RAW headline before normalization
df = df[~df[TEXT_COLUMN].fillna("").str.contains(BOILERPLATE_RE)]
n1 = len(df)

df[TEXT_COLUMN] = df[TEXT_COLUMN].map(normalize_headline)
df = df[df[TEXT_COLUMN].str.split().map(len) >= 4]            # require >= 4 words
n2 = len(df)

df[TIMESTAMP_COLUMN] = pd.to_datetime(df[TIMESTAMP_COLUMN]).dt.tz_localize(None)
df = df.sort_values(TIMESTAMP_COLUMN).reset_index(drop=True)
df[DOCUMENT_ID_COLUMN] = df.index
df[URL_COLUMN] = None
df[SOURCE_COLUMN] = df[SOURCE_COLUMN].fillna("unknown")
df["DerivedTickersId"] = df["DerivedTickersId"].fillna("")

print(f"After boilerplate blocklist: {n1:,}  (-{n0 - n1:,})")
print(f"After <4-word / date strip:  {n2:,}  (-{n1 - n2:,})")
print(f"Headlines used:              {len(df):,}")
print(f"Date span:                   {df[TIMESTAMP_COLUMN].min()} → {df[TIMESTAMP_COLUMN].max()}")
df.head(3)

Unique headlines in window: 21,658,899


After boilerplate blocklist: 58,233  (-1,767)
After <4-word / date strip:  53,581  (-4,652)
Headlines used:              53,581
Date span:                   2023-01-01 00:04:48.767000 → 2024-12-31 23:21:29.106000


,text,timestamp,source,DerivedTickersId,document_id,url
0,Vietnam’s abandoned F1 circuit finally hosts f...,2023-01-01 00:04:48.767,BLG,ALPRG@FP,0,None
1,"Time zone by time zone, world welcomes the new...",2023-01-01 01:24:02.319,NS1,3735162ZIM,1,None
2,TOP SPORTS STORIES for : Big Rock leads the wa...,2023-01-01 05:11:39.136,NS1,4480107Z;596856Z;1022737D,2,None


## 3. Embed once with FinLang

Embed the whole corpus a single time and hand the matrix to BERTrend (it re-uses rows per slice). With ~60k headlines this is the bulk of the runtime.

In [4]:
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

embeddings = embedder.encode(
    df[TEXT_COLUMN].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
assert len(embeddings) == len(df), "embeddings must align row-for-row with df"
print(f"Embeddings: {embeddings.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/838 [00:00<?, ?it/s]

Embeddings: (53581, 768)


## 4. Configure BERTopic + train across time slices

`BERTopicModel` reads a TOML config. After construction we **override the vectorizer** to add a finance-boilerplate stopword list (`inc, plc, ltd, ceo, results, …`) on top of the English stopwords — this is what kills the junk clusters at the representation level. `BERTrend.train_topic_models` then fits one model per slice and merges new topics into the cumulative set when cosine ≥ `MIN_SIMILARITY` (continuation), else creates a new theme (birth).

In [5]:
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

FINANCE_BOILERPLATE = {
    # corporate / governance boilerplate
    "inc", "plc", "ltd", "llc", "corp", "co", "sa", "ag", "nv", "group", "holdings",
    "ceo", "cfo", "coo", "cio", "chairman", "director", "board", "officer",
    "results", "report", "reports", "update", "announces", "announced", "announcement",
    "dividend", "agm", "egm", "shares", "stake", "stock", "stocks", "shareholders",
    "says", "said", "new", "year", "today", "week", "day", "amp", "via", "live",
    "review", "preview", "q1", "q2", "q3", "q4", "fy",
    # generic news filler that dominated the representations (the 'terrible labels')
    "good", "bad", "best", "better", "big", "way", "ways", "world", "global",
    "get", "gets", "make", "makes", "made", "take", "takes", "turn", "turns",
    "see", "sees", "seen", "hit", "hits", "set", "sets", "top", "back", "off",
    "total", "increase", "decrease", "change", "changes", "rise", "rises", "fall", "falls",
    "role", "call", "calls", "need", "needs", "plan", "plans", "thing", "things",
    "people", "man", "woman", "men", "women", "life", "home", "time", "times",
    "city", "limited", "release", "released", "expands", "replace", "phase",
    # sports / off-domain noise
    "win", "wins", "won", "loss", "losses", "season", "league", "cup", "match",
    "game", "games", "club", "team", "teams", "player", "players", "goal", "goals",
    "united", "sport", "sports",
}
CUSTOM_STOP_WORDS = list(ENGLISH_STOP_WORDS.union(FINANCE_BOILERPLATE))

bertopic_config = f"""
[global]
language = "English"

[bertopic_model]
top_n_words = 10
verbose = false
representation_model = ["MaximalMarginalRelevance"]
zeroshot_topic_list = []
zeroshot_min_similarity = 0

[umap_model]
n_neighbors = 15
n_components = 5
min_dist = 0.0
metric = "cosine"
random_state = {RANDOM_SEED}

[hdbscan_model]
min_cluster_size = {MIN_TOPIC_SIZE}
min_samples = {MIN_SAMPLES}
metric = "euclidean"
cluster_selection_method = "eom"
prediction_data = true

[vectorizer_model]
ngram_range = [1, 1]
stop_words = true
min_df = 3

[ctfidf_model]
bm25_weighting = false
reduce_frequent_words = true

[mmr_model]
diversity = 0.3

[reduce_outliers]
strategy = "c-tf-idf"
"""

topic_model = BERTopicModel(bertopic_config)
# Override the internal vectorizer with our extended finance stopword list.
topic_model.vectorizer_model = CountVectorizer(
    stop_words=CUSTOM_STOP_WORDS,
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",     # alpha tokens, >= 3 chars (drops tickers/numbers)
    ngram_range=(1, 2),                         # allow phrases ("interest rate", "clean energy")
    min_df=3,
)

# Use KeyBERTInspired to re-rank candidate words by embedding similarity to the topic
# (kills generic filler like "good/way/world") then MMR for diversity. We build the models
# directly because BERTrend's factory passes tuple args (nr_repr_docs=(5,)) that crash fit().
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance  # noqa: E402

topic_model.config["bertopic_model"]["representation_model"] = [
    KeyBERTInspired(top_n_words=20, nr_repr_docs=5, nr_candidate_words=40),
    MaximalMarginalRelevance(diversity=0.4),
]

# BERTrend._train_by_period calls fit() WITHOUT forwarding embedding_model, so BERTopic's
# embedding_model is None — fine for MMR but KeyBERTInspired needs it to embed candidate
# words/representative docs. Wrap fit to inject our FinLang embedder by default.
import functools  # noqa: E402

_orig_fit = topic_model.fit


@functools.wraps(_orig_fit)
def _fit_with_embedder(*args, **kwargs):
    kwargs.setdefault("embedding_model", embedder)
    return _orig_fit(*args, **kwargs)


topic_model.fit = _fit_with_embedder

bertrend = BERTrend(topic_model=topic_model)
bertrend.config["granularity"] = GRANULARITY_DAYS
bertrend.config["min_similarity"] = MIN_SIMILARITY

grouped_data = group_by_days(df=df, day_granularity=GRANULARITY_DAYS)
grouped_data = {ts: g for ts, g in grouped_data.items() if not g.empty}
print(f"Time slices: {len(grouped_data)}  (granularity = {GRANULARITY_DAYS}d)")
print(f"docs/slice — min {min(len(g) for g in grouped_data.values())}, "
      f"median {int(np.median([len(g) for g in grouped_data.values()]))}, "
      f"max {max(len(g) for g in grouped_data.values())}")

Time slices: 53  (granularity = 14d)
docs/slice — min 106, median 1027, max 1228


In [6]:
# This is the long step: ~one BERTopic fit per slice. Saved per-period so it is resumable.
bertrend.train_topic_models(
    grouped_data=grouped_data,
    embedding_model=embedder,
    embeddings=embeddings,
    bertrend_models_path=MODELS_DIR,
    save_topic_models=True,
)
# Detach the SentenceTransformer before pickling: injecting it for KeyBERT makes the
# trained model + our fit-wrapper hold an unpicklable sqlite3.Connection. None of the
# downstream steps (popularity, signals, Sankey) need the embedder.
topic_model.fit = _orig_fit
if bertrend.last_topic_model is not None:
    bertrend.last_topic_model.embedding_model = None
bertrend.save_model(models_path=MODELS_DIR)

print(f"Trained periods:  {len(bertrend.doc_groups)}")
print(f"Merged themes:    {bertrend.merged_df['Topic'].nunique() if bertrend.merged_df is not None else 0}")

2026-06-08 16:14:27.631 | INFO     | bertrend.BERTrend:train_topic_models:288 - Training topic model 1/53...


2026-06-08 16:14:27.632 | DEBUG    | bertrend.BERTrend:_train_by_period:186 - Processing period: 2023-01-01 00:00:00


2026-06-08 16:14:27.633 | DEBUG    | bertrend.BERTrend:_train_by_period:187 - Number of documents: 1018


2026-06-08 16:14:27.633 | DEBUG    | bertrend.BERTrend:_train_by_period:189 - Creating topic model...


2026-06-08 16:14:27.634 | DEBUG    | bertrend.BERTopicModel:fit:263 - 	Initializing BERTopic model


...

2026-06-08 16:16:17.319 | ERROR    | bertrend.BERTrend:train_topic_models:316 - Traceback:
Traceback (most recent call last):

  File "/Users/federicocinus/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
           │         │     └ {'__name__': '__main__', '__doc__': 'Entry point for launching an IPython kernel.\n\nThis is separate from the ipykernel pack...
           │         └ <code object <module> at 0x104919ce0, file "/Users/federicocinus/Progetti - Local/ThematicTrading/code/.venv/lib/python3.12/s...
           └ <function _run_code at 0x105399da0>
  File "/Users/federicocinus/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/runpy.py", line 88, in _run_code
    exec(code, run_globals)
         │     └ {'__name__': '__main__', '__doc__': 'Entry point for launching an IPython kernel.\n\nThis is separate from the ipykernel pack...
         └ <

2026-06-08 16:16:17.321 | ERROR    | bertrend.BERTrend:train_topic_models:323 - Training completed with failures for 8 periods: [Timestamp('2023-04-09 00:00:00'), Timestamp('2023-08-27 00:00:00'), Timestamp('2023-10-22 00:00:00'), Timestamp('2023-11-05 00:00:00'), Timestamp('2023-12-31 00:00:00'), Timestamp('2024-07-14 00:00:00'), Timestamp('2024-12-01 00:00:00'), Timestamp('2024-12-29 00:00:00')]


2026-06-08 16:16:17.322 | WARNING  | bertrend.BERTrend:train_topic_models:327 - Successfully trained topic models for 45 periods: [Timestamp('2023-01-01 00:00:00'), Timestamp('2023-01-15 00:00:00'), Timestamp('2023-01-29 00:00:00'), Timestamp('2023-02-12 00:00:00'), Timestamp('2023-02-26 00:00:00'), Timestamp('2023-03-12 00:00:00'), Timestamp('2023-03-26 00:00:00'), Timestamp('2023-04-23 00:00:00'), Timestamp('2023-05-07 00:00:00'), Timestamp('2023-05-21 00:00:00'), Timestamp('2023-06-04 00:00:00'), Timestamp('2023-06-18 00:00:00'), Timestamp('2023-07-02 00:00:00'), Timestamp('2023-07-16 00:00:00'), Timestamp('2023-07-30 00:00:00'), Timestamp('2023-08-13 00:00:00'), Timestamp('2023-09-10 00:00:00'), Timestamp('2023-09-24 00:00:00'), Timestamp('2023-10-08 00:00:00'), Timestamp('2023-11-19 00:00:00'), Timestamp('2023-12-03 00:00:00'), Timestamp('2023-12-17 00:00:00'), Timestamp('2024-01-14 00:00:00'), Timestamp('2024-01-28 00:00:00'), Timestamp('2024-02-11 00:00:00'), Timestamp('2024-02-

2026-06-08 16:16:17.523 | INFO     | bertrend.BERTrend:save_model:753 - BERTrend model saved to: /Users/federicocinus/Progetti - Local/ThematicTrading/code/notebooks/output/bertrend_models


Trained periods:  45
Merged themes:    18


## 5. Theme-intensity time series + **rate**  ← *the Stage-1 deliverable*

`calculate_signal_popularity` gives a per-theme popularity that **accumulates** while a theme keeps receiving docs (and decays when quiet). For the lead–lag test we also need a *rate* that peaks and falls, so we add:

- `intensity_rate` — first difference of popularity per theme (Δ popularity per slice),
- `new_docs` — non-cumulative docs per slice (Δ of BERTrend's cumulative `Docs_Count`).

The `new_docs` curve is the cleanest "attention pulse" for aligning against ETF price/volume peaks.

In [7]:
bertrend.calculate_signal_popularity()

# theme -> short keyword label from the merged topic representation
rep_map = {}
if bertrend.merged_df is not None:
    for _, r in bertrend.merged_df.iterrows():
        rep = r.get("Representation")
        rep_map[r["Topic"]] = ", ".join(rep[:5]) if isinstance(rep, (list, tuple)) else str(rep)


def theme_label(tid: int) -> str:
    return f"{tid}: {rep_map.get(tid, '')}"


rows = []
for tid, data in bertrend.topic_sizes.items():
    ts_list = data.get("Timestamps", [])
    pops = data.get("Popularity", [])
    docs_cum = data.get("Docs_Count", [0] * len(ts_list))
    for ts, pop, dc in zip(ts_list, pops, docs_cum):
        rows.append({
            "theme_id": tid, "timestamp": pd.Timestamp(ts),
            "intensity": float(pop), "docs_cum": float(dc), "label": theme_label(tid),
        })

intensity = pd.DataFrame(rows).sort_values(["theme_id", "timestamp"]).reset_index(drop=True)
g = intensity.groupby("theme_id")
intensity["intensity_rate"] = g["intensity"].diff().fillna(intensity["intensity"])
intensity["new_docs"] = g["docs_cum"].diff().fillna(intensity["docs_cum"]).clip(lower=0)

print(f"Themes tracked: {intensity['theme_id'].nunique()}   rows: {len(intensity)}")
intensity.head()

Themes tracked: 16   rows: 650


,theme_id,timestamp,intensity,docs_cum,label,intensity_rate,new_docs
0,0,2023-01-01,141.0,141.0,"0: partners, reach, japan, partner, american",141.0,141.0
1,0,2023-01-15,200.0,200.0,"0: partners, reach, japan, partner, american",59.0,59.0
2,0,2023-01-15,242.0,242.0,"0: partners, reach, japan, partner, american",42.0,42.0
3,0,2023-01-15,286.0,286.0,"0: partners, reach, japan, partner, american",44.0,44.0
4,0,2023-01-29,429.0,429.0,"0: partners, reach, japan, partner, american",143.0,143.0


In [8]:
# Cumulative intensity (top) vs per-slice attention pulse (bottom) for the top themes.
TOP_THEMES = 12
top_ids = (
    intensity.groupby("theme_id")["intensity"].max().sort_values(ascending=False).head(TOP_THEMES).index
)
plot_df = intensity[intensity["theme_id"].isin(top_ids)]

fig_int = px.line(
    plot_df, x="timestamp", y="intensity", color="label", markers=True,
    title=f"Cumulative theme-intensity — top {TOP_THEMES} themes ({TAG})",
)
fig_int.update_layout(legend_title_text="theme", height=520)
fig_int.write_html(OUTPUT_DIR / f"bertrend_intensity_{TAG}.html")
fig_int.show()

fig_rate = px.line(
    plot_df, x="timestamp", y="new_docs", color="label", markers=True,
    title=f"Attention pulse (new docs / slice) — top {TOP_THEMES} themes ({TAG})",
)
fig_rate.update_layout(legend_title_text="theme", height=520)
fig_rate.write_html(OUTPUT_DIR / f"bertrend_intensity_rate_{TAG}.html")
fig_rate.show()

## 6. Signal classification at the latest date

For `current_date`, BERTrend computes the P10/P50 popularity percentiles over the trailing `WINDOW_SIZE` days and buckets each theme into **noise / weak / strong**. Weak signals (emerging, rising) are the ones we care about for "before it's priced in".

In [9]:
current_date = max(bertrend.doc_groups.keys())
noise_df, weak_df, strong_df = bertrend.classify_signals(WINDOW_SIZE, current_date)

print(f"As of {pd.Timestamp(current_date).date()}  (window = {WINDOW_SIZE}d)")
print(f"  noise:  {len(noise_df)}")
print(f"  weak:   {len(weak_df)}   <- emerging themes")
print(f"  strong: {len(strong_df)}")

cols = [c for c in ["Topic", "Representation", "Latest_Popularity", "Docs_Count", "Sources"]
        if weak_df is not None and c in weak_df.columns]
if weak_df is not None and not weak_df.empty:
    display(weak_df[cols].head(10))
else:
    print("No weak signals at this date — try a different current_date or window.")

As of 2024-12-15  (window = 28d)
  noise:  8
  weak:   5   <- emerging themes
  strong: 3


,Topic,Representation,Latest_Popularity,Docs_Count,Sources
0,3,pharma_fda_drug_health_deal_insider_public_app...,3174.745603,3211,"{DBF, VOX, TIF, TIL, STS, SLE, OPA, PDA, FM5, ..."
1,7,news_nasa_march_expected_watch_night_public_mi...,1650.347842,1878,"{DBF, VOX, TIL, SLE, PDA, FM5, NS1, CO1, DML, ..."
2,8,company_share_bank_ipo_insider_tiktok_public_s...,4180.373388,4211,"{DBF, VRG, TIL, HUR, STS, PDA, FM5, NS1, CO1, ..."
3,11,growth_market_india_industry_energy_sales_cror...,1100.531024,1764,"{HUR, TIL, GBI, STS, OPA, PDA, FM5, NS1, CO1, ..."
4,15,christmas_buy_watch_free_sale_online_starbucks...,153.872366,313,"{FEX, TIL, TOI, BGL, STS, GO5, NS7, NS3, TGI, ..."


## 7. Map themes → ICB sectors **and** tickers

Two complementary mappings, since the H1 run showed the ICB-cosine map was unreliable (max cosine ≈ 0.5, AI→Leisure Goods, etc.):

1. **ICB cosine** — theme centroid vs the 45 ICB Sector centroids. We **calibrate** the `aligned` threshold from the empirical cosine distribution (75th percentile) instead of a hardcoded 0.30, and keep `sector_cos` as an explicit confidence score.
2. **Tickers** — aggregate the `DerivedTickersId` of each theme's member headlines. This is grounded in the data (not in ICB definitions) and is the more trustworthy bridge to Stage 3 exposure.

In [10]:
icb_raw = pd.read_excel(ICB_PATH, sheet_name="Mappable", header=1).dropna(subset=["Subsector"])
icb = pd.DataFrame({
    "industry": icb_raw["Industry"].str.strip(),
    "sector": icb_raw["Sector"].str.strip(),
    "subsector": icb_raw["Subsector"].str.strip(),
    "definition": icb_raw["Definition"].fillna("").str.strip(),
}).reset_index(drop=True)

sub_emb = embedder.encode(
    (icb["subsector"] + ". " + icb["definition"]).tolist(),
    batch_size=32, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True,
)
sector_meta = (
    icb.groupby("sector", as_index=False).agg(industry=("industry", "first"))
    .sort_values(["industry", "sector"]).reset_index(drop=True)
)
sector_emb = np.stack([sub_emb[(icb["sector"] == s).to_numpy()].mean(axis=0) for s in sector_meta["sector"]])
sector_emb /= np.linalg.norm(sector_emb, axis=1, keepdims=True)
print(f"Sector centroids: {sector_emb.shape}")

Sector centroids: (45, 768)


In [11]:
# --- ticker aggregation per theme (grounded in DerivedTickersId) ---
text2tick = dict(zip(df[TEXT_COLUMN], df["DerivedTickersId"]))


def _flatten_docs(documents) -> list[str]:
    """merged_df['Documents'] mixes flat strings and (timestamp, [docs]) tuples — flatten both."""
    out = []
    if not isinstance(documents, (list, tuple)):
        return out
    for item in documents:
        if isinstance(item, str):
            out.append(item)
        elif isinstance(item, tuple) and len(item) == 2 and isinstance(item[1], list):
            out.extend([t for t in item[1] if isinstance(t, str)])
        elif isinstance(item, list):
            out.extend([t for t in item if isinstance(t, str)])
    return out


def top_tickers(documents, k: int = 6) -> str:
    cnt = Counter()
    for t in _flatten_docs(documents):
        raw = text2tick.get(t, "")
        if isinstance(raw, str) and raw:
            for tk in raw.split(";"):
                tk = tk.strip()
                if tk:
                    cnt[tk] += 1
    return ", ".join(f"{tk}({c})" for tk, c in cnt.most_common(k))


docs_by_theme = dict(zip(bertrend.merged_df["Topic"], bertrend.merged_df["Documents"]))


def exemplar_headline(tid: int) -> str:
    """Pick the real headline that best represents a theme: the one overlapping the most
    theme keywords (ties broken toward shorter, punchier headlines). Far more legible than
    a bag of c-TF-IDF words."""
    kws = [w.strip().lower() for w in rep_map.get(tid, "").split(",") if w.strip()]
    docs = _flatten_docs(docs_by_theme.get(tid, []))
    if not docs:
        return ""
    best = max(docs, key=lambda h: (sum(kw in h.lower() for kw in kws), -len(h)))
    return best.strip()


# --- ICB cosine map (calibrated threshold) ---
theme_emb = np.stack(bertrend.merged_df["Embedding"].to_numpy())
theme_emb = theme_emb / (np.linalg.norm(theme_emb, axis=1, keepdims=True) + 1e-12)
sim = theme_emb @ sector_emb.T
nearest = sim.argmax(axis=1)
max_cos = sim.max(axis=1)

TAU_COV = float(np.round(np.percentile(max_cos, 75), 3))      # calibrated, not hardcoded
peak = intensity.groupby("theme_id")["intensity"].max()
peak_rate = intensity.groupby("theme_id")["new_docs"].max()

theme_ids = bertrend.merged_df["Topic"].to_numpy()
themes = pd.DataFrame({
    "theme_id": theme_ids,
    "keywords": [rep_map.get(t, "") for t in theme_ids],
    "sector_cos": np.round(max_cos, 3),
    "nearest_sector": sector_meta["sector"].to_numpy()[nearest],
    "nearest_industry": sector_meta["industry"].to_numpy()[nearest],
    "top_tickers": [top_tickers(docs_by_theme.get(t, [])) for t in theme_ids],
    "exemplar": [exemplar_headline(t) for t in theme_ids],
})
themes["description"] = themes.apply(
    lambda r: f'{r["keywords"]} — e.g. "{r["exemplar"]}"' if r["exemplar"] else r["keywords"],
    axis=1,
)
themes["sector_conf"] = np.where(themes["sector_cos"] >= TAU_COV, "aligned", "novel")
themes["peak_intensity"] = themes["theme_id"].map(peak).fillna(0.0)
themes["peak_new_docs"] = themes["theme_id"].map(peak_rate).fillna(0.0)
themes = themes.sort_values("peak_intensity", ascending=False).reset_index(drop=True)

tick_cov = (themes["top_tickers"].str.len() > 0).mean()
print(f"Calibrated TAU_COV (P75 of cosine): {TAU_COV}")
print(f"aligned: {(themes['sector_conf'] == 'aligned').sum()}   novel: {(themes['sector_conf'] == 'novel').sum()}")
print(f"themes with >=1 ticker: {tick_cov:.0%}")
themes.head(15)

Calibrated TAU_COV (P75 of cosine): 0.4259999990463257
aligned: 5   novel: 13
themes with >=1 ticker: 100%


,theme_id,keywords,sector_cos,nearest_sector,nearest_industry,top_tickers,exemplar,description,sector_conf,peak_intensity,peak_new_docs
0,2,"price target, price, ahead, downgrades, prices",0.336,"Gas, Water and Multi-utilities",Utilities,"%USD(323), %XBT(279), %JPY(278), %EUR(209), 83...",Analyst Downgrades Lucid Gr NASDAQLCID with Lo...,"price target, price, ahead, downgrades, prices...",novel,6533.000000,231.0
1,0,"partners, reach, japan, partner, american",0.427,"Gas, Water and Multi-utilities",Utilities,"3400398Z(172), AAPL(107), 0629846DBB(104), MSF...",Elys Game Technology partners with U.S. Integr...,"partners, reach, japan, partner, american — e....",aligned,6069.000000,278.0
2,1,"hydrogen, meeting, public, hong kong, conference",0.424,"Gas, Water and Multi-utilities",Utilities,"9904743Z(135), 0629846DBB(59), ISMEN@TI(53), 3...",Mar Conferences/Seminar Investor Meetings in S...,"hydrogen, meeting, public, hong kong, conferen...",novel,5632.000000,219.0
3,5,"chief, news, key, hotel, newly",0.307,Consumer Services,Consumer Discretionary,"AAPL(123), SPOT(84), 1022737D(60), 32663Z(54),...",Meet the chief marketing officers of US News' ...,"chief, news, key, hotel, newly — e.g. ""Meet th...",novel,4279.000000,393.0
4,8,"westpac, icici, billion, capital, exchange",0.466,Finance and Credit Services,Financials,"0235304Z(144), 9904743Z(64), 3400398Z(56), 883...",Westpac to raise $ million in additional Tier ...,"westpac, icici, billion, capital, exchange — e...",aligned,4180.373388,217.0
5,3,"certificate, details, county, act, safe",0.271,Non-life Insurance,Financials,"1000L(157), 1070L(109), 32663Z(75), GOOGL(62),...",Compliances-Reg. ( ) - Details of Loss of Cert...,"certificate, details, county, act, safe — e.g....",novel,3174.745603,191.0
6,4,"cuts, shortage, industry, sales, largest",0.400,Telecommunications Equipment,Telecommunications,"AAPL(342), MSFT(273), GOOGL(210), 1554630D(119...",Kia's Dec. sales rise pct despite chip shortage,"cuts, shortage, industry, sales, largest — e.g...",novel,3136.034950,153.0
7,9,"gop, legislature, senator, storms, state",0.289,"Gas, Water and Multi-utilities",Utilities,"1000L(153), 32663Z(120), AAPL(86), 0141541D(52...",New York state Legislature rejects proposed Ho...,"gop, legislature, senator, storms, state — e.g...",novel,2870.635854,377.0
8,7,"staff, message, warns, issue, fans",0.267,"Gas, Water and Multi-utilities",Utilities,"1022737D(193), 9889440Z(171), 0326786Z(152), A...","Man Utd Issue Update On Takeover, Send Message...","staff, message, warns, issue, fans — e.g. ""Man...",novel,1650.347842,127.0
9,6,"covid, passengers, longer, earlier, health",0.520,"Gas, Water and Multi-utilities",Utilities,"AAPL(59), 3400398Z(51), 0751538DSW(44), 817688...",announcement that the COVID- pandemic is no lo...,"covid, passengers, longer, earlier, health — e...",aligned,1597.621375,149.0


## 8. Persist (parquet)

In [12]:
intensity_path = OUTPUT_DIR / f"bertrend_intensity_{TAG}.parquet"
themes_path = OUTPUT_DIR / f"bertrend_themes_{TAG}.parquet"
intensity.to_parquet(intensity_path, index=False)
themes.to_parquet(themes_path, index=False)
print(f"Wrote {intensity_path}")
print(f"Wrote {themes_path}")

Wrote /Users/federicocinus/Progetti - Local/ThematicTrading/code/notebooks/output/bertrend_intensity_2023_2024.parquet
Wrote /Users/federicocinus/Progetti - Local/ThematicTrading/code/notebooks/output/bertrend_themes_2023_2024.parquet


## 9. Native BERTrend dashboards (3 HTML artifacts)

We render three standalone HTML dashboards mirroring the official BERTrend demos. A small helper concatenates Plotly figures + HTML tables into one self-contained page (Plotly JS via CDN, loaded once).

In [13]:
def write_dashboard(path: Path, title: str, blocks: list[tuple[str, object]]):
    """blocks: list of ('fig', go.Figure) | ('html', str) | ('h2', str) | ('p', str)."""
    parts, first_fig = [], True
    for kind, obj in blocks:
        if kind == "fig":
            parts.append(obj.to_html(full_html=False, include_plotlyjs="cdn" if first_fig else False))
            first_fig = False
        elif kind == "h2":
            parts.append(f"<h2>{obj}</h2>")
        elif kind == "p":
            parts.append(f"<p class='note'>{obj}</p>")
        elif kind == "html":
            parts.append(str(obj))
    style = (
        "body{font-family:-apple-system,Segoe UI,Arial,sans-serif;margin:0;padding:24px;"
        "background:#f4f6f9;color:#222} h1{color:#1f3a5f} h2{color:#2980b9;"
        "border-bottom:2px solid #d6e0ea;padding-bottom:6px;margin-top:32px}"
        ".note{color:#555;font-size:14px} .card{background:#fff;border-radius:8px;padding:18px;"
        "margin:14px 0;box-shadow:0 1px 3px rgba(0,0,0,.08)} table{border-collapse:collapse;"
        "font-size:13px;width:100%} th,td{border:1px solid #e2e8f0;padding:6px 8px;text-align:left}"
        "th{background:#eef3f8}"
    )
    html = (
        f"<!DOCTYPE html><html><head><meta charset='utf-8'><title>{title}</title>"
        f"<style>{style}</style></head><body><h1>{title}</h1>"
        + "<div class='card'>" + "</div><div class='card'>".join(parts) + "</div>"
        + "</body></html>"
    )
    Path(path).write_text(html)
    print(f"Wrote {path}")

### 9a. `topic_analysis` — per-slice topic exploration

Mirrors [`demos/topic_analysis`](https://github.com/rte-france/BERTrend/tree/main/bertrend/demos/topic_analysis): number of topics per slice, outlier size per slice, and the talked-about topics per news source for the latest slice.

In [14]:
from bertrend.trend_analysis.visualizations import (
    plot_num_topics,
    plot_size_outliers,
    plot_topics_for_model,
)

topic_models = bertrend.restore_topic_models(MODELS_DIR)        # dict[ts, BERTopic] (with doc_info_df)
latest_ts = max(topic_models)
latest_model = topic_models[latest_ts]

fig_ntopics = plot_num_topics(topic_models)
fig_outliers = plot_size_outliers(topic_models)
fig_src = plot_topics_for_model(latest_model)

write_dashboard(
    OUTPUT_DIR / f"bertrend_topic_analysis_{TAG}.html",
    f"BERTrend · Topic Analysis ({TAG})",
    [
        ("p", f"Per-slice topic models: {len(topic_models)} periods. Latest slice: {pd.Timestamp(latest_ts).date()}."),
        ("h2", "Number of topics detected per slice"), ("fig", fig_ntopics),
        ("h2", "Outlier (-1) size per slice"), ("fig", fig_outliers),
        ("h2", f"Topics per source — slice {pd.Timestamp(latest_ts).date()}"), ("fig", fig_src),
    ],
)
fig_ntopics.show()

2026-06-08 16:16:18,259 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


2026-06-08 16:16:18,260 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


2026-06-08 16:16:18,261 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


2026-06-08 16:16:18,263 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


2026-06-08 16:16:18,264 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


...

2026-06-08 16:16:18,303 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


2026-06-08 16:16:18,304 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


2026-06-08 16:16:18,305 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


2026-06-08 16:16:18,306 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


Wrote /Users/federicocinus/Progetti - Local/ThematicTrading/code/notebooks/output/bertrend_topic_analysis_2023_2024.html


### 9b. `weak_signals` — signal evolution + merge Sankey + emergence

Mirrors [`demos/weak_signals`](https://github.com/rte-france/BERTrend/tree/main/bertrend/demos/weak_signals): the popularity-evolution plot with **noise / weak / strong** shaded bands (log y-axis), the **merge Sankey** (continuation / birth across slices), and the **newly emerged** themes per timestamp.

In [15]:
from bertrend.trend_analysis.visualizations import (
    create_topic_size_evolution_figure,
    create_sankey_diagram_plotly,
    plot_newly_emerged_topics,
)

# Signal thresholds for the trailing window (P10 / P90 of popularity).
window_start, window_end, all_pop, q1, q3 = bertrend._compute_popularity_values_and_thresholds(
    WINDOW_SIZE, current_date
)


def signal_evolution_figure(q1: float, q3: float, current_date, top_n: int = 15):
    """Robust replacement for BERTrend's plot_topic_size_evolution.

    BERTrend's helper puts the y-axis on a log scale with range [0, y_max] — but Plotly
    expects log-axis ranges in log10 units, so [0, …] renders as 'NaN×10^Infinity' and the
    curves disappear. Here we keep a linear y-axis, show the full timeline, and shade the
    noise / weak / strong regions from the window thresholds.
    """
    top = (
        intensity.groupby("theme_id")["intensity"].max()
        .sort_values(ascending=False).head(top_n).index
    )
    d = intensity[intensity["theme_id"].isin(top)]
    y_max = max(float(d["intensity"].max()), float(q3), 1.0)

    fig = px.line(d, x="timestamp", y="intensity", color="label", markers=True)
    # signal-region bands
    fig.add_hrect(y0=0, y1=q1, fillcolor="rgba(128,128,128,0.12)", line_width=0,
                  annotation_text="noise", annotation_position="top left")
    fig.add_hrect(y0=q1, y1=q3, fillcolor="rgba(255,165,0,0.14)", line_width=0,
                  annotation_text="weak signal", annotation_position="top left")
    fig.add_hrect(y0=q3, y1=y_max * 1.1, fillcolor="rgba(0,200,0,0.12)", line_width=0,
                  annotation_text="strong signal", annotation_position="top left")
    # Plotly's add_vline does integer arithmetic on datetime x (buggy with pandas
    # Timestamps), so pass the x as epoch-milliseconds, which date axes use internally.
    cd_ms = pd.Timestamp(current_date).value // 10**6
    fig.add_vline(x=cd_ms, line_width=2, line_dash="dash", line_color="red",
                  annotation_text="current date", annotation_position="top")
    fig.update_layout(
        title="Lifetime cumulative popularity (BERTrend signal metric)",
        xaxis_title="Timestamp", yaxis_title="Popularity (cumulative doc count)",
        legend_title_text="theme", height=600,
    )
    return fig


def attention_pulse_figure(current_date, top_n: int = 15, smooth: int = 3):
    """The thematic-trading view: new documents per slice (a *flow*, not a stock).

    Unlike cumulative popularity this rises AND falls, so its peak can be lead-lag tested
    against the cohort's price/flow peaks (roadmap Stage 1 gate)."""
    top = (
        intensity.groupby("theme_id")["new_docs"].max()
        .sort_values(ascending=False).head(top_n).index
    )
    d = intensity[intensity["theme_id"].isin(top)].copy()
    d["pulse"] = (
        d.groupby("theme_id")["new_docs"]
        .transform(lambda s: s.rolling(smooth, min_periods=1, center=True).mean())
    )
    fig = px.line(d, x="timestamp", y="pulse", color="label", markers=True)
    cd_ms = pd.Timestamp(current_date).value // 10**6
    fig.add_vline(x=cd_ms, line_width=2, line_dash="dash", line_color="red",
                  annotation_text="current date", annotation_position="top")
    fig.update_layout(
        title=f"Attention pulse — new docs / slice ({smooth}-slice smoothed)",
        xaxis_title="Timestamp", yaxis_title="New documents per slice",
        legend_title_text="theme", height=600,
    )
    return fig


fig_pulse_main = attention_pulse_figure(current_date)
fig_evo = signal_evolution_figure(q1, q3, current_date)

# Merge Sankey (topic continuation / birth)
fig_sankey = create_sankey_diagram_plotly(bertrend.all_merge_histories_df)

# Newly emerged themes per timestamp
fig_births = (
    plot_newly_emerged_topics(bertrend.all_new_topics_df)
    if bertrend.all_new_topics_df is not None and not bertrend.all_new_topics_df.empty
    else go.Figure()
)

write_dashboard(
    OUTPUT_DIR / f"bertrend_weak_signals_{TAG}.html",
    f"BERTrend · Weak Signals ({TAG})",
    [
        ("p", f"As of {pd.Timestamp(current_date).date()} (window {WINDOW_SIZE}d). "
              "<b>Attention pulse</b> (new docs/slice) is the trading-relevant series — it "
              "rises and falls, so its peak can be lead-lag tested against price/flows. "
              "<b>Cumulative popularity</b> below is BERTrend's signal metric (a running total "
              "that only grows while a theme stays active, decaying only when it goes silent); "
              f"noise P10={q1:.1f}, strong P90={q3:.1f}."),
        ("h2", "Attention pulse — new docs / slice (trading view)"), ("fig", fig_pulse_main),
        ("h2", "Lifetime cumulative popularity (signal regions)"), ("fig", fig_evo),
        ("h2", "Topic merging — Sankey"), ("fig", fig_sankey),
        ("h2", "Newly emerged themes"), ("fig", fig_births),
    ],
)
fig_pulse_main.show()

Wrote /Users/federicocinus/Progetti - Local/ThematicTrading/code/notebooks/output/bertrend_weak_signals_2023_2024.html


### 9c. `prospective_demo` — point-in-time signal dashboard

Mirrors [`bertrend_apps/prospective_demo`](https://github.com/rte-france/BERTrend/tree/main/bertrend/bertrend_apps/prospective_demo): a decision-oriented snapshot at `current_date` — the classified **weak** and **strong** signal tables (with tickers joined from our theme map) plus the attention-pulse chart for the emerging themes.

In [16]:
tick_by_theme = dict(zip(themes["theme_id"], themes["top_tickers"]))
desc_by_theme = dict(zip(themes["theme_id"], themes["description"]))
sector_by_theme = dict(zip(themes["theme_id"], themes["nearest_sector"]))


def _signal_table(sig_df: pd.DataFrame) -> str:
    if sig_df is None or sig_df.empty:
        return "<p class='note'>none at this date</p>"
    keep = [c for c in ["Topic", "Latest_Popularity", "Docs_Count"] if c in sig_df.columns]
    t = sig_df[keep].copy()
    t.insert(1, "theme", t["Topic"].map(desc_by_theme).fillna(""))
    t["sector"] = t["Topic"].map(sector_by_theme).fillna("")
    t["tickers"] = t["Topic"].map(tick_by_theme).fillna("")
    return t.head(15).to_html(index=False, escape=False)


# attention pulse for the classified weak/strong themes
sig_ids = []
for d in (weak_df, strong_df):
    if d is not None and not d.empty and "Topic" in d.columns:
        sig_ids += d["Topic"].tolist()
pulse_df = intensity[intensity["theme_id"].isin(sig_ids)] if sig_ids else intensity[intensity["theme_id"].isin(top_ids)]
fig_pulse = px.line(
    pulse_df, x="timestamp", y="new_docs", color="label", markers=True,
    title="Attention pulse — classified weak/strong themes",
)
fig_pulse.update_layout(height=480, legend_title_text="theme")

write_dashboard(
    OUTPUT_DIR / f"bertrend_prospective_{TAG}.html",
    f"BERTrend · Prospective Dashboard ({TAG})",
    [
        ("p", f"Point-in-time snapshot as of {pd.Timestamp(current_date).date()} "
              f"(window {WINDOW_SIZE}d). Strong = established, Weak = emerging."),
        ("h2", f"Strong signals ({0 if strong_df is None else len(strong_df)})"),
        ("html", _signal_table(strong_df)),
        ("h2", f"Weak signals ({0 if weak_df is None else len(weak_df)})  ← emerging"),
        ("html", _signal_table(weak_df)),
        ("h2", "Attention pulse"), ("fig", fig_pulse),
    ],
)
fig_pulse.show()

Wrote /Users/federicocinus/Progetti - Local/ThematicTrading/code/notebooks/output/bertrend_prospective_2023_2024.html


## Next steps (Stage-1 gate)

- **Lead–lag test:** align each theme's **`new_docs` pulse** with the matching ETF cohort (e.g. Clean Energy = ICLN/TAN/QCLN…) and check the ordering `news-pulse → PX_LAST → PX_VOLUME / FUND_FLOW`. The lag is the edge.
- **Use the ticker map** (`themes.top_tickers`) as the candidate universe for Stage 3 exposure — more trustworthy than the ICB cosine here.
- **Tune merging:** sweep `MIN_SIMILARITY` and `GRANULARITY_DAYS` (try 7d); inspect the merge Sankey for over-merging.
- **Lifecycle labels:** derive birth / growth / peak / decay from the `new_docs` curve + signal class.

**Artifacts written to `notebooks/output/`:** `bertrend_intensity_{TAG}.html`, `bertrend_intensity_rate_{TAG}.html`, `bertrend_topic_analysis_{TAG}.html`, `bertrend_weak_signals_{TAG}.html`, `bertrend_prospective_{TAG}.html`, plus the two parquet files.